<a href="https://colab.research.google.com/github/prithivika2007/PRODIGY_GA_01/blob/main/Task01_TextGeneration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets torch

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from torch.utils.data import Dataset

# Custom Dataset class (replaces TextDataset)
class TextDataset(Dataset):
    def __init__(self, tokenizer, file_path, block_size=128):
        with open(file_path, "r") as f:
            text = f.read()
        tokenized = tokenizer(text, return_tensors="pt", truncation=False)["input_ids"][0]
        self.examples = [tokenized[i:i+block_size] for i in range(0, len(tokenized) - block_size, block_size)]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return {"input_ids": self.examples[i], "labels": self.examples[i]}

# Load GPT-2
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Load dataset
train_dataset = TextDataset(tokenizer, "data.txt", block_size=128)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Training settings
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=500,
    logging_steps=100,
)
# Train
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

trainer.train()
print("✅ Training done!")

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

result = generator(
    "The secret to success is",
    max_new_tokens=80,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.9,
    pad_token_id=tokenizer.eos_token_id
)

print(result[0]['generated_text'])

In [ ]:
prompts = [
    "Hard work and dedication will",
    "Believe in yourself because"
]

for p in prompts:
    result = generator(
        p,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    print(f"Prompt: {p}")
    print(result[0]['generated_text'])
    print("---")